In [ ]:
import pandas as pd
import requests
import re

# Caminho do arquivo .xls
arquivo = r"D:\Cotic\PLANILHA CONTROLE DE ALVARA 2023 A 2025_BEATRIZ_JURIDICO.xls"

df = pd.read_excel(arquivo, sheet_name="Planilha1", engine="xlrd")
ids_raw = df.iloc[:, 0].astype(str).str.strip()

ids = []
for val in ids_raw:
    if val and val.strip():
        try:
            ids.append(int(val))
            #ids.append(val)
        except ValueError:
            print(f"Valor não convertido: {val}")

print(f"Total de IDs capturados: {len(ids)}")

# Login
login_url = "http://172.18.4.46:8080/sispen_api/login/login"
login_payload = {"username": "elisangela.helcias", "password": "30069218"}
headers_login = {"Content-Type": "application/json"}

resp_login = requests.post(login_url, json=login_payload, headers=headers_login)
match = re.search(r"<token>(.*?)</token>", resp_login.text)
token = match.group(1)
TOKEN = f"Bearer {token}"

headers = {"Authorization": TOKEN, "Accept": "application/json"}

resultados = []

for pessoa_id in ids:
    try:
        #pessoa_id = re.sub(r"\.0$", "", pessoa_id) #só quando for a planilha da beatriz
        # 1. Obter id_real
        url_real = f"http://172.18.4.46:8080/sispen_api/custodiado?page=0&size=10&ativo=false&pessoaId={pessoa_id}"
        resp_real = requests.get(url_real, headers=headers)

        pessoa_nome = ""
        numero_processo = "Sem processo"

        if resp_real.status_code == 200:
            data_real = resp_real.json()
            entities_real = data_real.get("entities", [])
            if not entities_real:
                print(f"{pessoa_id} - Não encontrado")
                continue

            id_real = entities_real[0]["id"]
            pessoa_nome = entities_real[0].get("pessoaNome", "")
            id_pessoa_ficha = entities_real[0].get("pessoaId", pessoa_id)

            # Buscar processos
            url_processos = "http://172.18.4.46:8080/sispen_api/custodiado/ficha/processos"
            payload_processos = {
                "idPessoa": str(id_pessoa_ficha),   # <-- aqui está a correção
                "idUsuario": 3104,
                "idUnidade": 170,
                "idOrigem": 1,
                "unidadesAcesso": [170,157]
            }
            resp_processos = requests.post(url_processos, headers=headers, json=payload_processos)

            if resp_processos.status_code == 200:
                processos = resp_processos.json().get("processos", [])
                recolhidos = [
                    p for p in processos
                    if p.get("tipoSituacaoPrisional", {}).get("descricao", "").strip().lower() == "recolhido por este processo".lower()
                ]
                if recolhidos:
                    numero_processo = recolhidos[0].get("numeroProcesso", "")
                elif processos:
                    numero_processo = processos[0].get("numeroProcesso", "")

            saida_val = ""
            try:
                saida_val = df.loc[df.iloc[:,0].astype(str) == str(pessoa_id)].iloc[0,5]
            except:
                saida_val = ""


        resultados.append({
            "Prontuário": pessoa_id,
            "Processo": numero_processo,
            "Nome": pessoa_nome,
            "Saida": saida_val 
        })

        print(f"{pessoa_id} - Sucesso na requisição {numero_processo}")

    except Exception as e:
        print(f"{pessoa_id} - Erro na requisição: {e}")

df_final = pd.DataFrame(resultados)
saida = r"D:\Cotic\resultado_Processo_UPSOBRAL.xlsx"
df_final.to_excel(saida, index=False)
print(f"Planilha gerada em: {saida}")


Total de IDs capturados: 922
416378 - Sucesso na requisição 0200683-24.2022.8.06.0298
509965 - Sucesso na requisição Sem processo
469240 - Sucesso na requisição 0200386-25.2022.8.06.0069
393846 - Sucesso na requisição 0050029-30.2020.8.06.0028
484109 - Sucesso na requisição 0200236-02.2023.8.06.0298
205397 - Sucesso na requisição 0203027-46.2023.8.06.0167
473126 - Sucesso na requisição 0201770-15.2022.8.06.0298
216685 - Sucesso na requisição 98829-36.2015.8.06.0167
322633 - Erro na requisição: 'NoneType' object has no attribute 'get'
236198 - Erro na requisição: 'NoneType' object has no attribute 'get'
485820 - Sucesso na requisição 0201097-03.2023.8.06.0293
504140 - Sucesso na requisição 0203704-76.2023.8.06.0167
510659 - Sucesso na requisição Sem processo
498828 - Sucesso na requisição 0005285-92.2014.8.06.0081
507192 - Sucesso na requisição 0050665-34.2021.8.06.0101
502492 - Sucesso na requisição 0203312-34.2023.8.06.0298
508990 - Sucesso na requisição 0005838-12.2011.8.06.0028
4818

KeyboardInterrupt: 